# 用 YOLOv8 实现迁移学习 (Transfer Learning)

## 什么是迁移学习?

**迁移学习** 是指把在大型数据集(如 COCO,约 12 万张图、80 类)上**预训练好**的模型权重,复用到**自己的数据集**上继续训练。

YOLOv8 预训练时,网络的**浅层**已经学会了通用特征(边缘、纹理、形状),**深层**学会了"目标是什么"。迁移学习就是"站在巨人的肩膀上"——不用从零学起,只需在已有知识上微调。

## YOLOv8 中迁移学习的两种方式

### 方式一:直接微调 (Fine-tuning)—— 最常用
直接加载预训练权重 `yolov8n.pt`,在自己的数据集上继续训练。
- ✅ 训练快、收敛快、**数据量少也能训**
- 适用:任务与 COCO 相近(通用目标检测)

### 方式二:冻结层训练 (Freeze layers)
**冻结骨干网络(浅层通用特征)不更新**,只训练深层/检测头。
- ✅ 防止小数据集上"灾难性遗忘",更省显存、更稳
- 适用:数据量很少,或想保留 COCO 学到的通用特征

## 常用迁移学习参数

| 参数 | 作用 | 推荐值 |
|------|------|--------|
| `pretrained=True` | 使用预训练权重 | `True` |
| `freeze=10` | 冻结前 N 层(通常为 backbone) | `10` 或层索引列表 |
| `lr0` | 初始学习率(迁移学习用**较小值**) | `0.001 ~ 0.01` |
| `epochs` | 训练轮数 | `50 ~ 100`(小数据) |
| `data` | 数据集 yaml 路径 | 自定义 |

## ⚠️ 关键技巧:自定义数据集的类别数 ≠ 80 时

COCO 预训练模型是 **80 类**。如果你的数据集类别数不同,最后一层检测头(Detect head)的**结构会改变**,直接 `YOLO("yolov8n.pt")` 加载会权重错位/报错。

**正确做法(两段式):**
1. 先用 `yolov8n.yaml` **重建网络** → 检测头按你的类别数重新初始化
2. 再 `.load("yolov8n.pt")` 把**预训练权重**加载进去(backbone/neck 直接复用,检测头用新初始化)

> 也就是说:`YOLO("yolov8n.yaml").load("yolov8n.pt")` 才是"自定义数据集 + 迁移学习"的标准打开方式,而不是直接 `YOLO("yolov8n.pt")`。

## 常用迁移学习技巧总结

1. **小学习率**:`lr0=0.01` 以下,别让预训练权重被冲掉
2. **冻结浅层**:数据少时 `freeze=10`,先训检测头再解冻
3. **先冻结后解冻**:先用大 `freeze` 跑几轮,再 `freeze=0` 全量微调
4. **恢复训练**:中断后 `resume=True` 从 `last.pt` 接着训


In [ ]:
# ============================================================
# YOLOv8 迁移学习完整示例
# 前提:已安装 ultralytics,或用本地源码:
#   pip install -e ultralytics-8.4.113
# ============================================================
from ultralytics import YOLO

# ---------- 方式一:直接微调 (类别数与 COCO 相同或相近) ----------
# 自动下载 yolov8n.pt 预训练权重 (n=最小最快, 还有 s/m/l/x)
model = YOLO("yolov8n.pt")

model.train(
    data="coco8.yaml",      # ⚠️ 换成你自己的数据集 yaml (含 train/val 路径和 names)
    epochs=50,              # 迁移学习不需要太多轮次就能收敛
    imgsz=640,
    lr0=0.01,               # 迁移学习建议用较小学习率
    pretrained=True,        # 使用预训练权重 (默认 True)
    # freeze=10,            # 方式二:冻结前 10 层 (backbone) 后再训练
)

# ---------- 方式二:冻结层训练 (数据少 / 想保留通用特征) ----------
# model = YOLO("yolov8n.pt")
# model.train(data="coco8.yaml", epochs=50, freeze=10)   # 冻结前10层
# # 也可以传层索引列表,例如冻结 backbone 的指定层:
# # model.train(data="coco8.yaml", epochs=50, freeze=[0, 1, 2, 3, 4, 5])

# ---------- 自定义数据集 (类别数 ≠ 80) 的正确姿势 ----------
# 1) 用 yaml 重建网络 → 检测头按新类别数初始化
# 2) .load() 把预训练权重塞进去 → backbone/neck 直接复用
# model = YOLO("yolov8n.yaml").load("yolov8n.pt")
# model.train(
#     data="my_dataset.yaml",   # 自定义数据集: 比如 3 类水果
#     epochs=100,
#     imgsz=640,
#     lr0=0.005,
#     freeze=10,                # 数据少时先冻结 backbone 训检测头
# )

# ---------- 恢复中断的训练 ----------
# model = YOLO("runs/detect/train/weights/last.pt")
# model.train(resume=True)   # 自动接着上次的配置继续训练

# ---------- 训练完后用微调模型做推理 ----------
# best = YOLO("runs/detect/train/weights/best.pt")
# results = best("test.jpg")   # 或对视频/文件夹推理
